# Figuras do paper — *What implied volatility does to variance forecasters*

Cada pergunta de pesquisa tem **três opções de visualização**, 15 no total. A ideia é escolher uma ou duas por RQ para o texto principal e mandar o resto para o apêndice.

- **Fonte dos dados:** só os resultados finais em `results/` (Addenda C, D e E, com 2.000 sorteios de refit por célula). Nenhuma estimação é refeita aqui.
- **Saída:** cada figura é salva em `notebooks/figures/` como **PDF vetorial** (fontes embutidas, pronto para LaTeX) e **PNG a 600 dpi**.
- **Código de apoio:** estilo, cores e leitura de dados ficam em `notebooks/ivviz.py`. A paleta tem um único significado em todas as figuras:
  - **família:** HAR azul, elastic net laranja, boosted trees verde-azulado;
  - **componente:** teacher roxo, input carmim;
  - **placebo:** cinza;
  - **horizonte:** ▲ 5 sessões, ■ 10 sessões, ● 30 dias;
  - **índice:** marcador cheio = SPX, vazado = NDX.

| RQ | Opção A | Opção B | Opção C |
|---|---|---|---|
| RQ1 comportamento | curvas de resposta com fita de divergência | *ridgelines* do placebo espelhadas SPX/NDX | *horizon chart* de HFS e ISD no tempo |
| RQ2 acurácia 30d | *raincloud* dos ganhos (refit+datas) | ganho acumulado no tempo | ganho por estado (RV × surpresa) |
| RQ3 teacher × input | plano teacher–input com nuvens de refit | *slopegraph* da dependência da lei | cascata observado × placebo |
| RQ4 horizonte | *fan chart* da estrutura a termo | faixas de densidade de input − teacher | mapa horizonte × surpresa |
| RQ5 incerteza | migração dos p-valores | inflação dos intervalos + caudas do HAR | painel de calibração da simulação |

In [ ]:
%config InlineBackend.figure_format = "retina"
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mt
from matplotlib import patheffects as pe
from matplotlib.cm import ScalarMappable
from matplotlib.colors import LinearSegmentedColormap, Normalize, TwoSlopeNorm, to_rgb
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Patch, Rectangle
from matplotlib.transforms import blended_transform_factory
from scipy.stats import gaussian_kde

sys.path.insert(0, str(Path.cwd()))
import ivviz as V

V.set_style()
HALO = [pe.withStroke(linewidth=2.4, foreground="white")]
PCT = mt.PercentFormatter(1.0, decimals=0)

In [ ]:
# Gather the draw-level files once (refit+date: 18 cells x 2000 draws; placebo: 6 cells x 500 draws).
# The first run takes about a minute; later runs read notebooks/.cache/.
for a in V.ASSETS:
    for s in V.SPECS:
        for f in V.FAMILIES:
            V.refit_draws(a, s, f)
    for f in V.FAMILIES:
        V.placebo_draws(a, "primary", f)
print("draw cache ready:", len(list(V.CACHE.glob("*.parquet"))), "files")

---
## RQ1 — Incluir a IV muda o comportamento dos previsores?

### 1A · Curvas de resposta com fita de divergência
Grade 3 × 3 (família × variável do histórico) com as curvas ALE centradas de $f_0$ (preto), $f_1$ (cor da família) e da projeção $P_1$ (tracejada).
- **Fita entre $f_0$ e $f_1$:** a opacidade é proporcional ao afastamento local entre as duas curvas, então a fita "acende" onde a IV reorganiza a resposta.
- **Caixa de texto:** traz $\Delta S$ (×10⁻⁴, IC por bloco de datas) e $T$ em múltiplos do q95 do placebo. A borda fica colorida quando a regra P2 é afirmada.
- **Leitura:** em todas as 18 células (9 por índice) $\Delta S<0$, isto é, $f_1$ é **mais plana** que $f_0$: a IV assume parte do papel da persistência. Na maioria das células $P_1$ fica perto de $f_0$, então o achatamento vem da resposta à surpresa (ISD), não de uma nova função do histórico.
- **Caudas:** acima de q90 as curvas do HAR explodem. A escala vertical é ajustada à faixa central e a seta "tail" indica onde a curva sai do gráfico.

In [ ]:
def centred_curves(asset, family, feature):
    t = V.table(asset, "primary", f"response_curves_{family}.csv")
    t = t[t.feature == feature]
    P = t.pivot(index="edge", columns="function", values="value").sort_index()
    w = t[t.function == "le0"].set_index("edge")["bin_count"].reindex(P.index).fillna(0).to_numpy()
    return P.index.to_numpy(float), {m: P[m].to_numpy() - np.sum(w * P[m].to_numpy()) / w.sum()
                                     for m in ("le0", "le1", "le1p")}


def p2_cell(h, family, feat):
    g = h[h.family == family].set_index(["group", "statistic"])
    ds, T = g.loc[("P2", f"{feat}:dS")], g.loc[("P2_shape", f"{feat}:T")]
    claimed = bool((ds.ci_lo > 0 or ds.ci_hi < 0) and ds.boot_p_holm < 0.05 and T.placebo_p_holm < 0.05)
    return ds, T, claimed


def fig_response(asset):
    h = V.table(asset, "primary", "hierarchy.csv")
    fig, axes = plt.subplots(3, 3, figsize=(7.1, 6.0), sharex=True)
    S = 1e4
    for i, fam in enumerate(V.FAMILIES):
        col = V.FAM_COLOR[fam]
        for j, feat in enumerate(V.FEATS):
            ax = axes[i, j]
            x, c = centred_curves(asset, fam, feat)
            y0, y1, yp = c["le0"] * S, c["le1"] * S, c["le1p"] * S
            core = np.r_[y0[1:10], y1[1:10], yp[1:10]]
            pad = 0.45 * np.ptp(core)
            lo_, hi_ = core.min() - pad, core.max() + pad * 1.6      # room for the badge
            ax.axvspan(1, 9, color="#F6F6F6", lw=0, zorder=0)
            ax.axhline(0, color="#CCCCCC", lw=0.5, zorder=0)
            gap = np.abs(np.clip(y1, lo_, hi_) - np.clip(y0, lo_, hi_))
            V.gradient_between(ax, x, y0, y1, col, weight=gap, alpha=(0.04, 0.6))
            ax.plot(x, y0, color=V.INK, lw=1.15, zorder=4)
            ax.plot(x, yp, color=col, lw=0.95, ls=(0, (2.4, 1.6)), zorder=4)
            ax.plot(x, y1, color=col, lw=1.8, zorder=5, path_effects=HALO)
            for yy, face in ((y0, "white"), (y1, col)):
                ax.plot(x[[1, 9]], yy[[1, 9]], "o", ms=2.8, mfc=face, mec=V.INK if face == "white" else "white",
                        mew=0.7, zorder=6)
            ax.set_ylim(lo_, hi_)
            tails = [yy[10] for yy in (y0, y1, yp) if yy[10] > hi_ or yy[10] < lo_]
            if tails:
                up = max(tails, key=abs) > hi_
                ax.annotate("tail", xy=(10, hi_ if up else lo_), xytext=(9.25, hi_ - 0.1 * (hi_ - lo_) if up else lo_ + 0.1 * (hi_ - lo_)),
                            fontsize=5.4, color="#777777", ha="center", va="center", path_effects=HALO,
                            arrowprops=dict(arrowstyle="-|>", lw=0.5, color="#777777", shrinkA=0, shrinkB=0))
            ds, T, claimed = p2_cell(h, fam, feat)
            m_ = lambda v: f"{v:+.1f}".replace("-", "−")
            txt = (rf"$\Delta S$ {m_(ds.estimate * S)}  [{m_(ds.ci_lo * S)}, {m_(ds.ci_hi * S)}]" "\n"
                   rf"$T$ = {T.estimate / T.placebo_q95:.1f}$\times$ placebo q95")
            ax.text(0.035, 0.965, txt, transform=ax.transAxes, va="top", ha="left", fontsize=5.9, color=V.INK,
                    linespacing=1.35, zorder=8,
                    bbox=dict(boxstyle="round,pad=0.32,rounding_size=0.35", fc="white",
                              ec=col if claimed else "#BBBBBB", lw=0.8 if claimed else 0.5, alpha=0.93))
            ax.set_xticks([1, 5, 9], ["q10", "median", "q90"])
            ax.yaxis.set_major_locator(mt.MaxNLocator(4))
            ax.margins(x=0.02)
            if i == 0:
                ax.set_title(V.FEAT_LABEL[feat], loc="center", fontweight="normal", fontsize=8)
            if j == 0:
                ax.annotate(V.FAM_LABEL[fam], xy=(-0.3, 0.5), xycoords="axes fraction", rotation=90,
                            ha="center", va="center", fontsize=8.5, fontweight="bold", color=col)
    fig.supylabel(r"Centred accumulated effect on the forecast ($\times10^{-4}$)", x=0.008, fontsize=8)
    fig.supxlabel("Rank of the feature in the training window (decile edges)", y=0.035, fontsize=8)
    handles = [Line2D([], [], color=V.INK, lw=1.15, label=r"$f_0$  history only"),
               Line2D([], [], color="#6B6B6B", lw=1.8, label=r"$f_1$  history + IV"),
               Line2D([], [], color="#6B6B6B", lw=0.95, ls=(0, (2.4, 1.6)), label=r"$P_1$  IV integrated out"),
               Patch(fc="#8A8A8A", alpha=0.45, label=r"$f_1-f_0$  (opacity ∝ local gap)"),
               Patch(fc="#F6F6F6", ec="#CFCFCF", lw=0.5, label="central range q10–q90")]
    fig.legend(handles=handles, loc="upper center", ncol=5, bbox_to_anchor=(0.55, 0.965), columnspacing=1.2)
    fig.subplots_adjust(left=0.14, right=0.985, top=0.88, bottom=0.09, wspace=0.28, hspace=0.18)
    fig.text(0.14, 0.985, f"{asset}: how adding IV re-shapes the response to the volatility state",
             fontsize=9, fontweight="bold", ha="left", va="top")
    return fig


for asset in V.ASSETS:
    fig = fig_response(asset)
    V.save(fig, f"rq1a_response_ribbons_{asset}")
    plt.show()

### 1B · *Ridgelines* do placebo, espelhadas SPX/NDX
Cada linha é uma célula da regra P2 (família × variável).
- **Placebo:** a densidade cinza é a distância $T$ entre as curvas de resposta quando $f_1$ é reajustado numa **IV falsa com a mesma persistência** (500 sorteios). O SPX fica acima da linha de base e o NDX, espelhado, abaixo.
- **Escala:** o eixo x está em múltiplos da mediana do placebo, em escala log. A região escura marca os 5% superiores do placebo.
- **Observado:** o losango marca o $T$ observado (cheio = SPX, vazado = NDX).
- **Leitura:** todos os losangos caem além do placebo, nos dois índices. É a evidência visual de que a reorganização é específica da IV, e não de "qualquer variável persistente".

In [ ]:
def fig_placebo_ridges():
    rows = [(fam, feat) for fam in V.FAMILIES for feat in V.FEATS]
    obs = {a: V.table(a, "primary", "hierarchy.csv") for a in V.ASSETS}
    data = {}
    for fam, feat in rows:
        for a in V.ASSETS:
            pl = V.placebo_draws(a, "primary", fam)[f"{feat}:T"].to_numpy(float)
            h = obs[a]
            t_obs = h[(h.family == fam) & (h.group == "P2_shape") & (h.statistic == f"{feat}:T")].estimate.iloc[0]
            med = np.median(pl)
            data[(fam, feat, a)] = (np.log10(pl / med), np.log10(t_obs / med), np.log10(np.quantile(pl, 0.95) / med))
    hi = max(v[1] for v in data.values()) + 0.18
    grid = np.linspace(-0.75, hi, 700)
    fig, ax = plt.subplots(figsize=(6.6, 6.2))
    gap, amp = 1.0, 0.44
    for k, (fam, feat) in enumerate(rows):
        base = (len(rows) - 1 - k) * gap
        col = V.FAM_COLOR[fam]
        ax.axhline(base, color="#D9D9D9", lw=0.5, zorder=1)
        for a, sgn in (("SPX", 1), ("NDX", -1)):
            lr, lo, q95 = data[(fam, feat, a)]
            d = V.kde_1d(lr, grid, bw=0.25)
            d = d / d.max() * amp
            ax.fill_between(grid, base, base + sgn * d, color="#C7CBD1", lw=0, zorder=2, alpha=0.9)
            tail = grid >= q95
            ax.fill_between(grid[tail], base, base + sgn * d[tail], color="#7D848C", lw=0, zorder=3)
            ax.plot(grid, base + sgn * d, color="#6E747B", lw=0.5, zorder=3)
            ax.plot([lo, lo], [base, base + sgn * amp * 0.92], color=col, lw=1.1, zorder=4)
            ax.plot(lo, base + sgn * amp * 0.92, marker="D", ms=5.2, mfc=col if a == "SPX" else "white", mec=col,
                    mew=1.1, zorder=5)
            ax.text(hi + 0.02, base + sgn * 0.2, f"{10 ** lo:.1f}×", va="center", ha="left", fontsize=6.4, color=col,
                    fontweight="bold" if a == "SPX" else "normal")
        ax.text(-0.77, base, V.FEAT_LABEL[feat], ha="right", va="center", fontsize=7)
        if k % 3 == 1:
            ax.annotate(V.FAM_LABEL[fam], xy=(-0.205, base), xycoords=("axes fraction", "data"), rotation=90,
                        ha="center", va="center", fontsize=8, fontweight="bold", color=col)
        if k % 3 == 0 and k > 0:
            ax.axhline(base + gap / 2, color="#9E9E9E", lw=0.6, ls=(0, (1, 2)), zorder=1)
    ax.axvline(0, color="#9E9E9E", lw=0.6, ls="--", zorder=1)
    ticks = [t for t in (0.25, 0.5, 1, 2, 4, 8, 16, 32) if np.log10(t) <= hi]
    ax.set_xticks(np.log10(ticks), [{0.25: "¼×", 0.5: "½×"}.get(t, f"{t:g}×") for t in ticks])
    ax.set_xlim(-0.75, hi)
    ax.set_ylim(-0.6, (len(rows) - 1) * gap + 0.6)
    ax.set_yticks([])
    ax.spines["left"].set_visible(False)
    ax.set_xlabel(r"Response-curve distance $T$, in multiples of the placebo median (log scale)")
    ax.text(hi + 0.02, (len(rows) - 1) * gap + 0.62, "observed /\nplacebo median", fontsize=6.2, color="#555555",
            ha="left", va="bottom")
    handles = [Patch(fc="#C7CBD1", label="placebo: fake IV with the same persistence (500 refits)"),
               Patch(fc="#7D848C", label="placebo top 5%"),
               Line2D([], [], marker="D", ls="", mfc="#555555", mec="#555555", label="observed, SPX (above line)"),
               Line2D([], [], marker="D", ls="", mfc="white", mec="#555555", label="observed, NDX (below line)")]
    ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.45, 1.09), ncol=2, columnspacing=1.4)
    return fig


fig = fig_placebo_ridges()
V.save(fig, "rq1b_placebo_ridgelines")
plt.show()

### 1C · *Horizon chart* da reorganização ao longo do tempo
Para cada índice e família há duas faixas: $\mathrm{HFS}/f_0$ (reorganização da função do histórico) e $\mathrm{ISD}/f_0$ (resposta à surpresa da IV), em média móvel de 5 sessões.
- **Como ler um *horizon chart*:** a magnitude é "dobrada" em três camadas de 10 pontos percentuais. Cores quentes indicam que a IV **eleva** a previsão e cores frias que a **reduz**, então muita informação cabe em pouca altura.
- **Faixa superior de cada índice:** mostra a surpresa da IV, $\hat\varepsilon_t$ (lei A).
- **Leitura:**
  - a ISD acompanha a surpresa quase dia a dia (quente quando a IV sobe além do esperado);
  - a HFS é mais fraca e mais persistente;
  - os episódios de estresse concentram as maiores intensidades.
- **Faixas em branco no pico da COVID:** não são dados faltando. O estado saiu do suporte de treino e os modelos que cortam as entradas no intervalo de treino (elastic net, árvores) deixam de reagir à IV, então a ISD fica ≈ 0.

In [ ]:
def fig_horizon(band=0.10, smooth=5):
    comps = [("HFS_a", "HFS"), ("ISD_a", "ISD")]
    ratios = []
    for a in V.ASSETS:
        ratios += [0.7] + [1.0] * 6 + ([0.55] if a == "SPX" else [])
    fig = plt.figure(figsize=(7.1, 6.4))
    gs = GridSpec(len(ratios), 1, figure=fig, height_ratios=ratios, hspace=0.08,
                  left=0.13, right=0.985, top=0.93, bottom=0.12)
    r = 0
    axes = []
    snorm = TwoSlopeNorm(0, -2.5, 2.5)
    for a in V.ASSETS:
        base = V.origin(a, "primary", "har").sort_values("date")
        dates = mdates.date2num(base.date)
        ax = fig.add_subplot(gs[r]); r += 1
        eps = base.eps_a.rolling(smooth, center=True, min_periods=1).mean().to_numpy()
        ax.imshow(eps[None, :], aspect="auto", cmap=V.SURPRISE_CMAP, norm=snorm,
                  extent=[dates[0], dates[-1], 0, 1], interpolation="nearest")
        ax.set_yticks([])
        ax.text(-0.012, 0.5, f"{a}  IV surprise", transform=ax.transAxes, ha="right", va="center",
                fontsize=7.2, fontweight="bold")
        axes.append(ax)
        for fam in V.FAMILIES:
            o = V.origin(a, "primary", fam).sort_values("date")
            for key, name in comps:
                ax = fig.add_subplot(gs[r], sharex=axes[0]); r += 1
                rel = (o[key] / o.f0).rolling(smooth, center=True, min_periods=1).mean().to_numpy()
                V.horizon(ax, mdates.date2num(o.date), rel, band)
                ax.set_yticks([])
                ax.text(-0.012, 0.5, f"{V.FAM_LABEL[fam]}  {name}", transform=ax.transAxes, ha="right",
                        va="center", fontsize=6.6, color=V.FAM_COLOR[fam])
                axes.append(ax)
        if a == "SPX":
            sp = fig.add_subplot(gs[r]); r += 1
            sp.axis("off")
    for ax in axes:
        for s in ("left", "top", "right"):
            ax.spines[s].set_visible(False)
        ax.spines["bottom"].set_color("#E0E0E0")
        ax.tick_params(axis="x", length=0, labelbottom=False)
        for a0, _, _ in V.EPISODES:
            ax.axvline(mdates.date2num(pd.Timestamp(a0)), color="#555555", lw=0.45, ls=(0, (1, 1.6)), zorder=9)
    axes[-1].tick_params(axis="x", length=2.5, labelbottom=True)
    axes[-1].xaxis.set_major_locator(mdates.YearLocator())
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    axes[0].set_xlim(mdates.date2num(pd.Timestamp("2017-01-01")), mdates.date2num(pd.Timestamp("2025-12-05")))
    for a0, b0, name in V.EPISODES:
        mid = mdates.date2num(pd.Timestamp(a0) + (pd.Timestamp(b0) - pd.Timestamp(a0)) / 2)
        axes[0].text(mid, 1.12, name.replace("\n", " "), transform=axes[0].get_xaxis_transform(), ha="center",
                     va="bottom", fontsize=5.8, color="#444444")
    # legends: horizon bands and surprise colour bar
    pos = ("#FDD9B5", "#F4A261", "#C8553D"); neg = ("#CFE3F2", "#7FB3D8", "#2C6E9E")
    hl = ([Patch(fc=c, label=l) for c, l in zip(pos, ["+0 to 10%", "+10 to 20%", "> +20%"])]
          + [Patch(fc=c, label=l) for c, l in zip(neg, ["−0 to 10%", "−10 to 20%", "< −20%"])])
    fig.legend(handles=hl, loc="lower left", bbox_to_anchor=(0.12, 0.0), ncol=6, columnspacing=0.9, handlelength=1.1,
               title=r"Component / $f_0$  (warm: IV raises the forecast, cool: lowers it)", title_fontsize=6.8)
    cax = fig.add_axes([0.80, 0.035, 0.17, 0.012])
    cb = fig.colorbar(ScalarMappable(snorm, V.SURPRISE_CMAP), cax=cax, orientation="horizontal")
    cb.set_ticks([-2, 0, 2]); cb.outline.set_linewidth(0.4); cb.ax.tick_params(labelsize=6, length=2)
    cb.set_label(r"IV surprise $\hat\varepsilon_t$", fontsize=6.8, labelpad=1)
    fig.text(0.13, 0.975, "When and how IV moves the forecasts: history-function shift (HFS) vs. response to the "
             "IV surprise (ISD)", fontsize=8.6, fontweight="bold")
    return fig


fig = fig_horizon()
V.save(fig, "rq1c_horizon_hfs_isd")
plt.show()

---
## RQ2 — A mudança melhora a previsão de 30 dias?

As figuras das RQ2–RQ5 usam a largura de texto de um template de coluna única (7 pol.) e altura de 2,3 a 2,9 pol. Nos eixos, as famílias aparecem abreviadas: **HAR**, **EN** (elastic net) e **BT** (boosted trees).

### 2A · *Raincloud* vertical dos ganhos relativos (seis células confirmatórias)
Cada coluna é uma célula (índice × família); o painel (a) mostra o ganho total e o painel (b), o ganho input, ambos relativos a $L(f_0)$.
- **Meia-densidade à direita:** os 2.000 sorteios refit+datas.
- **Barras coloridas:** os intervalos de 80% e 95% desses sorteios, com o losango na estimativa pontual.
- **Barra preta fina (à esquerda):** o IC de 95% só por datas.
- **Traço horizontal preto:** o limite unilateral de 95% (q95), usado na regra C3. A faixa verde acima de 5% é a margem; um ganho ≥ 5% só é descartado se o traço ficar abaixo dela.
- **Quadrado vazado (painel b):** o mesmo ganho com um GARCH só de preços no lugar da IV.
- **Leitura:** as nuvens atravessam o zero e também passam da margem de 5%. Logo, **não há ganho detectável e um ganho moderado também não pode ser descartado**.

In [ ]:
def fig_raincloud(YL=(-0.30, 0.25)):
    items = [it for it in V.cell_grid(["primary"])]
    fig, axes = plt.subplots(1, 2, figsize=(V.W, 2.75), sharey=True)
    grid = np.linspace(*YL, 400)
    titles = {"total": r"a   Total gain  $[L(f_0)-L(f_1)]\,/\,L(f_0)$",
              "input": r"b   Input gain  $[L(P_1)-L(f_1)]\,/\,L(f_0)$"}
    for ax, comp in zip(axes, ("total", "input")):
        ax.axhspan(0.05, YL[1], color="#EEF6F4", lw=0, zorder=0)
        ax.axhline(0, color="#8A8A8A", lw=0.6, zorder=1)
        ax.axhline(0.05, color="#3F9C8F", lw=0.7, ls=(0, (1.2, 1.6)), zorder=1)
        for x, s, a, f in items:
            col = V.FAM_COLOR[f]
            d = V.refit_draws(a, s, f)[f"rel_gain_{comp}"].to_numpy(float)
            dens = V.kde_1d(d[(d > YL[0]) & (d < YL[1])], grid)
            dens = dens / dens.max() * 0.38
            on = dens > 0.012
            ax.fill_betweenx(grid, x + 0.05, x + 0.05 + dens, where=on, color=col, alpha=0.28, lw=0, zorder=2)
            ax.plot(np.where(on, x + 0.05 + dens, np.nan), grid, color=col, lw=0.6, zorder=3)
            for q, lw in ((0.95, 0.9), (0.80, 2.8)):
                lo, hi = np.quantile(d, [(1 - q) / 2, 1 - (1 - q) / 2])
                ax.plot([x - 0.07] * 2, [max(lo, YL[0]), min(hi, YL[1])], color=col, lw=lw, solid_capstyle="butt", zorder=3)
            b = V.boot_stat(a, s, f, f"rel_gain_{comp}")
            ax.errorbar(x - 0.24, b.estimate, yerr=[[b.estimate - b.ci_lo], [b.ci_hi - b.estimate]], fmt="none",
                        ecolor=V.INK, elinewidth=0.6, capsize=1.3, capthick=0.6, zorder=3)
            ax.plot(x - 0.07, b.estimate, "D", ms=3.4, mfc="white", mec=V.INK, mew=0.7, zorder=5)
            q95 = np.quantile(d, 0.95)
            ax.plot([x - 0.16, x + 0.02], [min(q95, YL[1])] * 2, color=V.INK, lw=1.1, zorder=5)
            low = (d < YL[0]).mean()
            if low > 0.005:
                ax.text(x + 0.12, YL[0] + 0.006, f"↓{low:.0%}", ha="left", va="bottom", fontsize=5.3, color=col)
            if comp == "input":
                g = V.sec_row(a, "garch", f)
                ax.errorbar(x - 0.38, g.rel_gain_input, yerr=[[g.rel_gain_input - g.rel_gain_input_lo],
                                                              [g.rel_gain_input_hi - g.rel_gain_input]],
                            fmt="s", ms=2.6, mfc="white", mec="#7A7A7A", mew=0.6, ecolor="#A0A0A0", elinewidth=0.5,
                            capsize=0.9, zorder=4)
        ax.set_ylim(*YL)
        ax.yaxis.set_major_formatter(PCT)
        ax.set_xlim(items[0][0] - 0.6, items[-1][0] + 0.6)
        V.multilevel_xlabels(ax, items, horizon=False, y_asset=-0.1)
        ax.set_title(titles[comp], fontsize=7.6)
        V.soft_grid(ax, "y")
    handles = [Patch(fc="#9A9A9A", alpha=0.4, label="refit+date draws"),
               Line2D([], [], color="#555555", lw=2.8, label="80% / 95% interval"),
               Line2D([], [], color=V.INK, lw=0.6, marker="_", ms=4, label="95% date-only"),
               Line2D([], [], marker="D", ls="", mfc="white", mec=V.INK, ms=3.4, label="estimate"),
               Line2D([], [], color=V.INK, lw=1.1, label="one-sided 95% bound"),
               Line2D([], [], marker="s", ls="", mfc="white", mec="#7A7A7A", ms=3, label="GARCH instead of IV"),
               Line2D([], [], color="#3F9C8F", lw=0.8, ls=(0, (1.2, 1.6)), label="C3 margin (5%)")]
    fig.legend(handles=handles, loc="lower center", ncol=7, bbox_to_anchor=(0.53, -0.01), columnspacing=0.8,
               handletextpad=0.35, handlelength=1.4, fontsize=5.9)
    fig.subplots_adjust(left=0.075, right=0.995, top=0.9, bottom=0.25, wspace=0.05)
    return fig


fig = fig_raincloud()
V.save(fig, "rq2a_raincloud_gains")
plt.show()

### 2B · Ganho acumulado no tempo
Os painéis SPX e NDX ficam lado a lado. Cada curva é a soma acumulada de $L(f_0)-L(f_1)$: sobe quando a IV ajuda e desce quando atrapalha.
- **Barras na borda direita:** o IC de 95% (bootstrap de datas) do ponto final.
- **Legenda:** traz o ganho médio de cada família em % de $L(f_0)$.
- **Áreas cinza:** os episódios de estresse.
- **Leitura:**
  - no SPX, ganhos e perdas vêm em **saltos concentrados** (fev-2018, COVID) e se anulam;
  - no NDX a trajetória é mais favorável, mas o IC do ponto final ainda inclui o zero para HAR e EN.

In [ ]:
def fig_cumulative():
    fig, axes = plt.subplots(1, 2, figsize=(V.W, 2.45))
    short = {"Feb-2018\nvol shock": "Feb-18", "COVID\ncrash": "COVID", "2022\nbear market": "2022",
             "Apr-2025\ntariffs": "Apr-25"}
    for ax, a in zip(axes, V.ASSETS):
        for a0, b0, name in V.EPISODES:
            ax.axvspan(pd.Timestamp(a0), pd.Timestamp(b0), color="#EDEDED", lw=0, zorder=0)
            ax.text(pd.Timestamp(a0) + (pd.Timestamp(b0) - pd.Timestamp(a0)) / 2, 1.01, short[name],
                    transform=ax.get_xaxis_transform(), ha="center", va="bottom", fontsize=5.4, color="#666666")
        ax.axhline(0, color="#7A7A7A", lw=0.6, zorder=1)
        handles = []
        for k, f in enumerate(V.FAMILIES):
            o = V.origin(a, "primary", f).sort_values("date")
            g = np.asarray(V.qlike(o.y, o.f0) - V.qlike(o.y, o.f1), float)
            cum = np.cumsum(g)
            ax.plot(o.date, cum, color=V.FAM_COLOR[f], lw=1.2, zorder=3,
                    path_effects=[pe.Stroke(linewidth=2.4, foreground="white"), pe.Normal()])
            b = V.boot_stat(a, "primary", f, "gain_total")
            n = len(g)
            xe = o.date.iloc[-1] + pd.Timedelta(days=80 + 85 * k)
            ax.errorbar(xe, cum[-1], yerr=[[cum[-1] - n * b.ci_lo], [n * b.ci_hi - cum[-1]]], fmt="o", ms=2.6,
                        color=V.FAM_COLOR[f], mfc="white", mew=0.8, elinewidth=0.9, capsize=1.4, clip_on=False, zorder=5)
            rel = cum[-1] / n / V.sec_row(a, "primary", f).L0
            handles.append(Line2D([], [], color=V.FAM_COLOR[f], lw=1.4, label=V.minus(f"{V.FAM_LABEL[f]}  {rel:+.1%}")))
        ax.legend(handles=handles, loc="lower left" if a == "SPX" else "upper left", fontsize=6,
                  title=r"mean gain, % of $L(f_0)$", title_fontsize=6, alignment="left")
        ax.set_title(a, loc="center", fontsize=8, pad=9)
        ax.xaxis.set_major_locator(mdates.YearLocator(2))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        ax.set_xlim(pd.Timestamp("2016-12-01"), pd.Timestamp("2026-08-01"))
        V.soft_grid(ax, "y")
    axes[0].set_ylabel(r"cumulative $\sum_t\,[L(f_0)-L(f_1)]$")
    fig.subplots_adjust(left=0.085, right=0.99, top=0.86, bottom=0.11, wspace=0.2)
    return fig


fig = fig_cumulative()
V.save(fig, "rq2b_cumulative_gain")
plt.show()

### 2C · Ganho por estado (RV histórica × surpresa da IV)
São seis pequenas matrizes 3 × 3 em linha (SPX à esquerda, NDX à direita): o tercil da RV histórica (vertical) contra o tercil da surpresa da IV (horizontal), com L/M/H = baixo/médio/alto.
- **Cor:** o ganho total relativo dentro do estado.
- **Tamanho do quadrado:** proporcional ao número de datas.
- **Leitura:** o ganho **muda de sinal entre estados**, e efeitos opostos se anulam na média.
  - no NDX a IV ajuda sobretudo quando a surpresa é alta;
  - no SPX o padrão é misto, com perdas grandes do EN quando a RV histórica é alta.

In [ ]:
def fig_state_tiles(vmax=0.35):
    norm = TwoSlopeNorm(0, -vmax, vmax)
    rv_rows, iv_cols = ["rv_low", "rv_mid", "rv_high"], ["iv_low", "iv_mid", "iv_high"]
    fig = plt.figure(figsize=(V.W, 1.75))
    gs = GridSpec(1, 7, figure=fig, width_ratios=[1, 1, 1, 0.22, 1, 1, 1], wspace=0.1,
                  left=0.06, right=0.9, top=0.8, bottom=0.14)
    slots = [0, 1, 2, 4, 5, 6]
    axes = []
    for k, (a, f) in enumerate([(a, f) for a in V.ASSETS for f in V.FAMILIES]):
        ax = fig.add_subplot(gs[slots[k]])
        axes.append(ax)
        st = V.table(a, "primary", f"teacher_states_{f}.csv")
        inner = st[(st.rv_state != "all") & (st.surprise != "all")]
        nmax = inner.n.max()
        for r, rv in enumerate(rv_rows):
            for c, iv in enumerate(iv_cols):
                row = inner[(inner.rv_state == rv) & (inner.surprise == iv)].iloc[0]
                val = row.gain_total / row.L0
                side = 0.95 * np.sqrt(row.n / nmax)
                z = norm(np.clip(val, -vmax, vmax))
                ax.add_patch(FancyBboxPatch((c - side / 2, r - side / 2), side, side, boxstyle="round,pad=0,rounding_size=0.08",
                                            fc=V.GAIN_CMAP(z), ec="white", lw=0.6))
                ax.text(c, r, V.fmt_pct(val), ha="center", va="center", fontsize=5.6,
                        color="white" if abs(z - 0.5) > 0.3 else V.INK)
        ax.set_xlim(-0.55, 2.55); ax.set_ylim(-0.55, 2.55); ax.set_aspect("equal")
        ax.set_xticks([0, 1, 2], ["L", "M", "H"])
        ax.set_yticks([0, 1, 2], ["L", "M", "H"] if k in (0, 3) else [])
        ax.tick_params(length=0, labelsize=6, pad=1)
        for s_ in ax.spines.values():
            s_.set_visible(False)
        ax.set_title(V.FAM_LABEL[f], loc="center", color=V.FAM_COLOR[f], fontsize=7, pad=2)
    fig.canvas.draw()                                   # equal-aspect axes settle their final box here
    top, bot = axes[0].get_position().y1, axes[0].get_position().y0
    for i, a in enumerate(V.ASSETS):
        p0, p1 = axes[3 * i].get_position(), axes[3 * i + 2].get_position()
        xm = (p0.x0 + p1.x1) / 2
        fig.text(xm, top + 0.2, a, ha="center", va="center", fontsize=8.5, fontweight="bold")
        fig.add_artist(Line2D([p0.x0, p1.x1], [top + 0.15] * 2, color="#9E9E9E", lw=0.6))
    fig.text(0.48, bot - 0.2, "IV-surprise tercile", ha="center", va="center", fontsize=7)
    fig.text(0.012, (top + bot) / 2, "historical RV tercile", rotation=90, va="center", fontsize=7)
    cax = fig.add_axes([0.925, bot, 0.012, top - bot])
    cb = fig.colorbar(ScalarMappable(norm, V.GAIN_CMAP), cax=cax, extend="both")
    cb.set_ticks([-0.3, 0, 0.3]); cb.ax.yaxis.set_major_formatter(PCT)
    cb.outline.set_linewidth(0.4); cb.ax.tick_params(labelsize=6, length=2)
    cb.set_label(r"total gain, % of $L(f_0)$", fontsize=6.4)
    return fig


fig = fig_state_tiles()
V.save(fig, "rq2c_state_tiles")
plt.show()

---
## RQ3 — O ganho vem de a IV *ensinar* o histórico ou de ser observada em tempo real?

### 3A · O plano teacher–input
Cada ponto é uma célula: a cor indica a família e o marcador o horizonte (▲ 5 sessões, ■ 10 sessões, ● 30 dias). O eixo x é o ganho teacher e o eixo y, o ganho input, ambos relativos a $L(f_0)$.
- **Cruzes finas:** o intervalo interquartil dos 2.000 sorteios refit+datas em cada eixo.
- **× cinza ligado por linha pontilhada:** a média do placebo, ou seja, onde a célula estaria com uma IV falsa persistente.
- **Fundo e linhas tracejadas:** o ganho total (x + y).
- **Outros marcadores:** a estrela é a âncora HAR-OLS e os círculos vazados são as redes neurais (só datas).
- **Leitura:** as células de horizonte curto sobem no eixo input sem andar no eixo teacher, e ficam longe do placebo. O teacher fica perto de zero.

In [ ]:
def fig_plane(XL=(-0.10, 0.09), YL=(-0.09, 0.16)):
    fig, axes = plt.subplots(1, 2, figsize=(V.W, 2.95), sharey=True)
    gx, gy = np.meshgrid(np.linspace(*XL, 200), np.linspace(*YL, 200))
    tnorm = TwoSlopeNorm(0, -0.22, 0.22)
    for ax, a in zip(axes, V.ASSETS):
        ax.imshow(gx + gy, extent=[*XL, *YL], origin="lower", cmap=V.GAIN_CMAP, norm=tnorm, alpha=0.22,
                  aspect="auto", zorder=0)
        cs = ax.contour(gx, gy, gx + gy, levels=[-0.05, 0.05, 0.1], colors="#9A9A9A", linewidths=0.4,
                        linestyles="--", zorder=1)
        ax.clabel(cs, fmt=lambda v: V.minus(f"total {v:+.0%}"), fontsize=5, inline=True, inline_spacing=1)
        ax.axhline(0, color="#6E6E6E", lw=0.5, zorder=1); ax.axvline(0, color="#6E6E6E", lw=0.5, zorder=1)
        for s in V.SPECS:
            for f in V.FAMILIES:
                col = V.FAM_COLOR[f]
                d, r = V.refit_draws(a, s, f), V.sec_row(a, s, f)
                x0, y0 = r.rel_gain_teacher, r.rel_gain_input
                qx = np.quantile(d.rel_gain_teacher, [0.25, 0.75]); qy = np.quantile(d.rel_gain_input, [0.25, 0.75])
                ax.plot(qx, [y0, y0], color=col, lw=0.8, alpha=0.55, zorder=3, solid_capstyle="butt")
                ax.plot([x0, x0], qy, color=col, lw=0.8, alpha=0.55, zorder=3, solid_capstyle="butt")
                px, py = r.rel_gain_teacher_placebo_mean, r.rel_gain_input_placebo_mean
                ax.plot([px, x0], [py, y0], color="#9A9A9A", lw=0.5, ls=(0, (1, 1.4)), zorder=2)
                ax.plot(px, py, "x", ms=2.8, color="#8A8A8A", mew=0.6, zorder=3)
                ax.plot(x0, y0, V.HORIZON_MARKER[s], ms=5.2, mfc=col, mec="white", mew=0.6, zorder=5)
        r = V.sec_row(a, "primary", "har_ols")
        tx, ty = r.rel_gain_teacher, r.rel_gain_input
        cx, cy = np.clip(tx, XL[0] + 0.004, XL[1] - 0.004), np.clip(ty, YL[0] + 0.005, YL[1] - 0.005)
        ax.plot(cx, cy, "*", ms=8, mfc=V.FAM_COLOR["har_ols"], mec=V.INK, mew=0.5, zorder=6)
        if (cx, cy) != (tx, ty):
            ax.annotate(V.minus(f"HAR-OLS off scale\n({tx:+.0%}, {ty:+.0%})"), (cx, cy), xytext=(cx - 0.045, cy + 0.02),
                        fontsize=5.3, color="#444444", arrowprops=dict(arrowstyle="-", lw=0.4, color="#666666"),
                        path_effects=HALO, zorder=7)
        for f in ("mlp", "tcn"):
            r = V.sec_row(a, "primary", f, profile="nn2")
            ax.plot(r.rel_gain_teacher, r.rel_gain_input, "o", ms=4.6, mfc="none", mec=V.FAM_COLOR[f], mew=0.9, zorder=5)
        for tx_, ty_, txt, ha in ((0.02, 0.97, "input only", "left"), (0.98, 0.03, "teacher only", "right")):
            ax.text(tx_, ty_, txt, transform=ax.transAxes, ha=ha, va="top" if ty_ > 0.5 else "bottom", fontsize=5.6,
                    color="#777777", style="italic")
        ax.set_xlim(*XL); ax.set_ylim(*YL)
        ax.xaxis.set_major_locator(mt.MultipleLocator(0.05)); ax.yaxis.set_major_locator(mt.MultipleLocator(0.05))
        ax.xaxis.set_major_formatter(PCT); ax.yaxis.set_major_formatter(PCT)
        ax.set_title(a, loc="center", fontsize=8)
        ax.set_xlabel(r"teacher gain  $[L(f_0)-L(P_1)]\,/\,L(f_0)$", fontsize=7)
    axes[0].set_ylabel(r"input gain  $[L(P_1)-L(f_1)]\,/\,L(f_0)$", fontsize=7)
    handles = ([Line2D([], [], marker="o", ls="", mfc=V.FAM_COLOR[f], mec="white", ms=5, label=V.FAM_LABEL[f])
                for f in V.FAMILIES]
               + [Line2D([], [], marker=V.HORIZON_MARKER[s], ls="", mfc="#6B6B6B", mec="white", ms=5,
                         label=V.HORIZON[s]) for s in V.SPECS]
               + [Line2D([], [], marker="x", ls=(0, (1, 1.4)), color="#8A8A8A", lw=0.6, ms=3, label="placebo mean"),
                  Line2D([], [], marker="*", ls="", mfc=V.FAM_COLOR["har_ols"], mec=V.INK, ms=6, label="HAR-OLS"),
                  Line2D([], [], marker="o", ls="", mfc="none", mec=V.FAM_COLOR["mlp"], ms=4.5, label="MLP / TCN")])
    fig.legend(handles=handles, loc="lower center", ncol=9, bbox_to_anchor=(0.53, -0.01), columnspacing=0.8,
               handletextpad=0.25, fontsize=6)
    fig.subplots_adjust(left=0.085, right=0.99, top=0.92, bottom=0.25, wspace=0.05)
    return fig


fig = fig_plane()
V.save(fig, "rq3a_teacher_input_plane")
plt.show()

### 3B · *Slopegraph* da dependência da lei condicional
São quatro painéis em linha (SPX e NDX, em 30 dias e 5 sessões). Cada linha liga o ganho teacher da mesma célula sob quatro versões da projeção:
- **Law A:** a lei A;
- **A on B-supp.:** a lei A restrita ao suporte da lei B;
- **Law B:** a lei kernel;
- **A, recal.:** a lei A após a recalibração de nível em tempo real.

As barras verticais são os ICs de 95% (bootstrap de datas). A linha clara grossa é a **âncora HAR-OLS**, cujo ganho teacher é exatamente zero sob uma lei linear.
- **Leitura:** a distância da âncora até zero (cerca de 5% de $L(f_0)$ sob a lei A) mede o viés da própria lei. Ganhos teacher desse tamanho **não se distinguem de artefato da lei**.

In [ ]:
def fig_slopes():
    variants = [("gain_teacher", "Law A"), ("gain_teacher_lawA_on_supB", "A on\nB-supp."),
                ("gain_teacher_lawB", "Law B"), ("gain_teacher_recal", "A,\nrecal.")]
    panels = [("SPX", "primary"), ("NDX", "primary"), ("SPX", "h5td"), ("NDX", "h5td")]
    fig, axes = plt.subplots(1, 4, figsize=(V.W, 2.45))
    for ax, (a, s) in zip(axes, panels):
        ax.axhline(0, color="#7A7A7A", lw=0.6, zorder=1)
        fams = V.FAMILIES + (["har_ols"] if s == "primary" else [])
        pts = []
        for k, f in enumerate(fams):
            r = V.sec_row(a, s, f)
            ys = np.array([r[v] / r.L0 for v, _ in variants])
            pts += list(ys)
            xs = np.arange(4) + (k - (len(fams) - 1) / 2) * 0.07
            for x, (v, _) in zip(xs, variants):
                lo, hi = r.get(f"{v}_lo"), r.get(f"{v}_hi")
                if pd.notna(lo):
                    ax.plot([x, x], [lo / r.L0, hi / r.L0], color=V.FAM_COLOR[f], lw=0.5, alpha=0.5, zorder=2)
            anchor = f == "har_ols"
            ax.plot(xs, ys, "-o", color=V.FAM_COLOR[f], lw=2.4 if anchor else 1.1, ms=3.2,
                    mfc="white" if anchor else V.FAM_COLOR[f], mec=V.FAM_COLOR[f], mew=0.9, zorder=4 if anchor else 3)
        lo, hi = min(pts), max(pts)
        pad = 0.25 * (hi - lo) + 0.01
        ax.set_ylim(lo - pad, hi + pad)
        ax.set_xlim(-0.35, 3.35)
        ax.set_xticks(range(4), [v[1] for v in variants], fontsize=5.7)
        ax.tick_params(axis="x", length=0)
        ax.yaxis.set_major_formatter(PCT)
        ax.yaxis.set_major_locator(mt.MaxNLocator(5, steps=[1, 2, 5, 10]))
        ax.set_title(f"{a} · {V.HORIZON[s]}", loc="center", fontsize=7.2)
        V.soft_grid(ax, "y")
    axes[0].set_ylabel(r"teacher gain, % of $L(f_0)$", fontsize=7)
    handles = ([Line2D([], [], color=V.FAM_COLOR[f], marker="o", ms=3, lw=1.1, label=V.FAM_LABEL[f]) for f in V.FAMILIES]
               + [Line2D([], [], color=V.FAM_COLOR["har_ols"], marker="o", mfc="white", ms=3, lw=2.4,
                         label="HAR-OLS anchor (0 in theory)")])
    fig.legend(handles=handles, loc="lower center", ncol=4, bbox_to_anchor=(0.53, -0.01), fontsize=6.3)
    fig.subplots_adjust(left=0.08, right=0.995, top=0.9, bottom=0.27, wspace=0.38)
    return fig


fig = fig_slopes()
V.save(fig, "rq3b_law_slopegraph")
plt.show()

### 3C · Cascata observado × placebo
As colunas aparecem em pares, por horizonte, índice e família: a da esquerda é o **observado** (sólida) e a da direita, a **média do placebo** (hachurada).
- **Empilhamento:** a parte roxa é o teacher e a carmim, o input, empilhada sobre ela. O traço preto marca o ganho total.
- **Linha cinza:** o intervalo 5–95% do ganho total sob o placebo.
- **Leitura:**
  - em 5 e 10 sessões o input observado **supera em muito** o do placebo, que fica perto de zero;
  - em 30 dias, onde há teacher grande (SPX EN, NDX BT), o placebo reproduz boa parte dele: é regularização genérica, não informação de opções.

In [ ]:
def fig_waterfall():
    items = V.cell_grid(V.SPECS, gap_asset=0.45, gap_spec=1.1)
    fig, ax = plt.subplots(figsize=(V.W, 2.75))
    w = 0.36
    ax.axhline(0, color="#7A7A7A", lw=0.6, zorder=1)
    for x, s, a, f in items:
        r = V.sec_row(a, s, f)
        pl = V.placebo_summary(a, s, f, "rel_gain_total")
        for t, i, xx, obs in ((r.rel_gain_teacher, r.rel_gain_input, x - 0.2, True),
                              (r.rel_gain_teacher_placebo_mean, r.rel_gain_input_placebo_mean, x + 0.2, False)):
            kw = dict(width=w, lw=0.5, zorder=3)
            if obs:
                ax.bar(xx, t, bottom=0, color=V.COMP_COLOR["teacher"], ec="white", **kw)
                ax.bar(xx, i, bottom=t, color=V.COMP_COLOR["input"], ec="white", **kw)
            else:
                ax.bar(xx, t, bottom=0, color="white", ec=V.COMP_COLOR["teacher"], hatch="//////", **kw)
                ax.bar(xx, i, bottom=t, color="white", ec=V.COMP_COLOR["input"], hatch="//////", **kw)
            ax.plot([xx - w / 2 - 0.03, xx + w / 2 + 0.03], [t + i] * 2, color=V.INK, lw=1.1, zorder=5,
                    solid_capstyle="butt")
        ax.plot([x + 0.2] * 2, [pl.placebo_q05, pl.placebo_q95], color="#8C8C8C", lw=0.8, zorder=4)
    V.horizon_separators(ax, items)
    ax.set_ylim(-0.125, 0.19)
    ax.set_xlim(items[0][0] - 0.6, items[-1][0] + 0.6)
    ax.yaxis.set_major_formatter(PCT)
    ax.set_ylabel(r"% of $L(f_0)$", fontsize=7)
    V.multilevel_xlabels(ax, items, y_asset=-0.1, y_hor=-0.19)
    V.soft_grid(ax, "y")
    handles = [Patch(fc=V.COMP_COLOR["teacher"], label="teacher"), Patch(fc=V.COMP_COLOR["input"], label="input"),
               Patch(fc="white", ec="#555555", hatch="//////", label="placebo mean (right bar)"),
               Line2D([], [], color=V.INK, lw=1.1, label="total"),
               Line2D([], [], color="#8C8C8C", lw=0.8, label="placebo 5–95% of total")]
    ax.legend(handles=handles, loc="upper right", ncol=5, fontsize=6, bbox_to_anchor=(1.0, 1.12), columnspacing=1.0)
    fig.subplots_adjust(left=0.075, right=0.995, top=0.9, bottom=0.24)
    return fig


fig = fig_waterfall()
V.save(fig, "rq3c_waterfall_placebo")
plt.show()

---
## RQ4 — O valor da IV depende do horizonte?

### 4A · *Fan chart* da estrutura a termo do ganho input
O eixo x é o horizonte em sessões, em escala log. Para cada família há um "leque" com as faixas de 50%, 80% e 95% dos 2.000 sorteios refit+datas do ganho input relativo; o círculo é a estimativa pontual.
- **Faixa superior:** traz $P(\text{sorteio}>0)$.
- **Setas embaixo:** a fração de sorteios fora da escala.
- **Leitura:**
  - o ganho input é mais alto em 5 sessões (7–11% de $L(f_0)$) e decai até ≈ 0 em 30 dias no SPX;
  - no NDX ele decai menos, e a maior parte dos sorteios fica acima de zero até 10 sessões;
  - a cauda dos sorteios (sobretudo no HAR) impede qualquer afirmação depois do Holm.

In [ ]:
def fig_fan(YL=(-0.16, 0.28), cap=0.222):
    fig, axes = plt.subplots(1, 2, figsize=(V.W, 2.6), sharey=True)
    offs = {"har": 0.88, "enet": 1.0, "xgb": 1.135}
    for ax, a in zip(axes, V.ASSETS):
        ax.set_xscale("log")
        ax.axhline(0, color="#7A7A7A", lw=0.6, zorder=1)
        ax.axhspan(0, cap, color="#F2F8F7", lw=0, zorder=0)
        for f in V.FAMILIES:
            xs, ys = [], []
            for s in V.SPECS:
                x = V.HORIZON_SESSIONS[s] * offs[f]
                d = V.refit_draws(a, s, f)["rel_gain_input"].to_numpy(float)
                for q, lw, al in ((0.95, 1.1, 0.35), (0.80, 3.0, 0.6), (0.50, 5.2, 0.95)):
                    lo, hi = np.quantile(d, [(1 - q) / 2, 1 - (1 - q) / 2])
                    ax.plot([x, x], [max(lo, YL[0]), min(hi, cap)], color=V.FAM_COLOR[f], lw=lw, alpha=al,
                            solid_capstyle="butt", zorder=3)
                pt = V.sec_row(a, s, f).rel_gain_input
                ax.plot(x, pt, "o", ms=3.6, mfc="white", mec=V.FAM_COLOR[f], mew=1.0, zorder=6)
                ax.text(x, 0.252, f"{(d > 0).mean():.0%}", ha="center", va="center", fontsize=5.3,
                        color=V.FAM_COLOR[f], fontweight="bold", zorder=7)
                low = (d < YL[0]).mean()
                if low > 0.005:
                    ax.annotate(f"{low:.0%}", xy=(x, YL[0]), xytext=(x, YL[0] + 0.03), ha="center", fontsize=5,
                                color=V.FAM_COLOR[f], arrowprops=dict(arrowstyle="-|>", lw=0.5, color=V.FAM_COLOR[f]))
                xs.append(x); ys.append(pt)
            ax.plot(xs, ys, color=V.FAM_COLOR[f], lw=0.7, ls=(0, (3, 2)), alpha=0.8, zorder=2)
        ax.axhline(cap + 0.007, color="#D0D0D0", lw=0.5, zorder=5)
        ax.set_xticks([5, 10, 21], ["5 sessions", "10 sessions", "30 days"])
        ax.xaxis.set_minor_locator(mt.NullLocator())
        ax.set_xlim(3.9, 27)
        ax.set_ylim(*YL)
        ax.set_yticks(np.arange(-0.15, 0.21, 0.05))
        ax.yaxis.set_major_formatter(PCT)
        ax.set_title(a, loc="center", fontsize=8)
        V.soft_grid(ax, "y")
    axes[0].text(-0.02, 0.252, r"$P(>0)$", transform=blended_transform_factory(axes[0].transAxes, axes[0].transData),
                 ha="right", va="center", fontsize=6, color="#444444")
    axes[0].set_ylabel(r"input gain, % of $L(f_0)$", fontsize=7)
    handles = ([Line2D([], [], color=V.FAM_COLOR[f], lw=3.5, label=V.FAM_LABEL[f]) for f in V.FAMILIES]
               + [Line2D([], [], color="#6B6B6B", lw=5.2, alpha=0.95, label="50%"),
                  Line2D([], [], color="#6B6B6B", lw=3.0, alpha=0.6, label="80%"),
                  Line2D([], [], color="#6B6B6B", lw=1.1, alpha=0.35, label="95% of refit+date draws")])
    fig.legend(handles=handles, loc="lower center", ncol=6, bbox_to_anchor=(0.53, -0.01), columnspacing=1.1,
               fontsize=6.3)
    fig.subplots_adjust(left=0.09, right=0.995, top=0.91, bottom=0.21, wspace=0.05)
    return fig


fig = fig_fan()
V.save(fig, "rq4a_term_structure_fan")
plt.show()

### 4B · Colunas de densidade do contraste input − teacher
Cada coluna é uma célula (horizonte × índice × família) e mostra a distribuição, entre os 2.000 sorteios refit+datas, de $G_{input}-G_{teacher}$ relativo, a quantidade testada pela regra D3.
- **Coluna:** a intensidade da cor segue a densidade (um "*gradient interval*").
- **Sobreposições:** o traço preto é o IC de 95% e o círculo é a mediana.
- **Linhas de texto acima:** $P(>0)$ e o p de Holm da D3. A D3 só existe para 5 e 10 sessões; em 30 dias o p, entre parênteses, é não ajustado.
- **Leitura:** em 5 sessões a maior parte da massa fica do lado positivo (input > teacher) nas seis células, com $P(>0)$ entre 66% e 99%, mas nenhuma passa o Holm sobre 12 células. Em 10 sessões o quadro é misto.

In [ ]:
def fig_strips(YL=(-0.14, 0.22)):
    items = V.cell_grid(V.SPECS, gap_asset=0.45, gap_spec=1.1)
    cD = V.claims_D().set_index(["asset", "spec", "family"])
    fig, ax = plt.subplots(figsize=(V.W, 2.85))
    ax.axhspan(0, YL[1], color="#F4F9F8", lw=0, zorder=0)
    ax.axhline(0, color="#6E6E6E", lw=0.6, zorder=1)
    tr = blended_transform_factory(ax.transData, ax.transAxes)
    for x, s, a, f in items:
        d = V.refit_draws(a, s, f)
        v = (d.rel_gain_input - d.rel_gain_teacher).to_numpy(float)
        V.density_strip_v(ax, x, v, V.FAM_COLOR[f], YL, width=0.66, bw=0.15)
        lo, med, hi = np.quantile(v, [0.025, 0.5, 0.975])
        ax.plot([x, x], [max(lo, YL[0]), min(hi, YL[1])], color=V.INK, lw=0.6, zorder=3)
        ax.plot(x, med, "o", ms=3, mfc="white", mec=V.INK, mew=0.7, zorder=4)
        if lo < YL[0]:
            ax.text(x, YL[0] + 0.002, "↓", ha="center", va="bottom", fontsize=5.5, color=V.INK)
        ax.text(x, 1.04, f"{(v > 0).mean():.0%}", transform=tr, ha="center", va="bottom", fontsize=5.3)
        if s == "primary":
            ptxt, bold = f"({V.fmt_p(V.boot_p(v))})", False
        else:
            p = cD.loc[(a, s, f)].imt_p_holm
            ptxt, bold = V.fmt_p(p), p < 0.05
        ax.text(x, 1.12, ptxt, transform=tr, ha="center", va="bottom", fontsize=5.3,
                fontweight="bold" if bold else "normal", color="#777777" if s == "primary" else V.INK)
    left = blended_transform_factory(ax.transAxes, ax.transAxes)
    ax.text(-0.01, 1.04, r"$P(>0)$", transform=left, ha="right", va="bottom", fontsize=5.8)
    ax.text(-0.01, 1.12, "Holm p (D3)", transform=left, ha="right", va="bottom", fontsize=5.8)
    V.horizon_separators(ax, items)
    ax.set_ylim(*YL)
    ax.set_xlim(items[0][0] - 0.6, items[-1][0] + 0.6)
    ax.yaxis.set_major_formatter(PCT)
    ax.set_ylabel("input − teacher,\n" + r"% of $L(f_0)$", fontsize=7)
    V.multilevel_xlabels(ax, items, y_asset=-0.1, y_hor=-0.19)
    fig.subplots_adjust(left=0.1, right=0.995, top=0.8, bottom=0.22)
    return fig


fig = fig_strips()
V.save(fig, "rq4b_density_strips_d3")
plt.show()

### 4C · Mapa horizonte × surpresa da IV
Linhas = índice × família. As colunas formam três blocos (5 sessões, 10 sessões, 30 dias), cada um com os tercis de surpresa da IV (baixa, média, alta).
- **Cor e número:** o ganho input relativo dentro do estado.
- **Leitura:** o valor da IV em tempo real vem **das surpresas extremas** (altas e, no SPX em 5 sessões, também baixas) e quase nada da surpresa média. Ele é mais forte no curto prazo; em 30 dias, mesmo as surpresas altas rendem pouco no SPX.

In [ ]:
def fig_horizon_surprise(vmax=0.3):
    rows = [(a, f) for a in V.ASSETS for f in V.FAMILIES]
    ivs = ["iv_low", "iv_mid", "iv_high"]
    norm = TwoSlopeNorm(0, -vmax, vmax)
    fig, ax = plt.subplots(figsize=(V.W, 2.35))
    xs = [j + (j // 3) * 0.4 for j in range(9)]
    ys = [-(i + (0.35 if i >= 3 else 0)) * 0.78 for i in range(len(rows))]
    for i, (a, f) in enumerate(rows):
        for g, s in enumerate(V.SPECS):
            st = V.table(a, s, f"teacher_states_{f}.csv")
            st = st[st.rv_state == "all"].set_index("surprise")
            for k, iv in enumerate(ivs):
                v = st.loc[iv, "gain_input"] / st.loc[iv, "L0"]
                z = norm(np.clip(v, -vmax, vmax))
                x, y = xs[3 * g + k], ys[i]
                ax.add_patch(FancyBboxPatch((x - 0.46, y - 0.34), 0.92, 0.68, boxstyle="round,pad=0,rounding_size=0.08",
                                            fc=V.GAIN_CMAP(z), ec="white", lw=0.6))
                ax.text(x, y, V.fmt_pct(v), ha="center", va="center", fontsize=6,
                        color="white" if abs(z - 0.5) > 0.3 else V.INK)
    ax.set_xticks(xs, ["low", "mid", "high"] * 3, fontsize=6)
    ax.set_yticks(ys, [f"{a} · {V.FAM_LABEL[f]}" for a, f in rows], fontsize=6.3)
    for lab, (a, f) in zip(ax.get_yticklabels(), rows):
        lab.set_color(V.FAM_COLOR[f])
    for g, s in enumerate(V.SPECS):
        xc = np.mean(xs[3 * g:3 * g + 3])
        ax.text(xc, 0.62, V.HORIZON[s], ha="center", va="bottom", fontsize=7.4, fontweight="bold")
        ax.plot([xs[3 * g] - 0.42, xs[3 * g + 2] + 0.42], [0.55, 0.55], color="#9E9E9E", lw=0.6)
    ax.set_xlim(-0.6, xs[-1] + 0.6)
    ax.set_ylim(ys[-1] - 0.5, 0.95)
    ax.tick_params(length=0, pad=2)
    for s_ in ax.spines.values():
        s_.set_visible(False)
    ax.set_xlabel("IV-surprise tercile", fontsize=6.8, labelpad=2)
    cax = fig.add_axes([0.93, 0.2, 0.011, 0.6])
    cb = fig.colorbar(ScalarMappable(norm, V.GAIN_CMAP), cax=cax, extend="both")
    cb.set_ticks([-0.3, -0.15, 0, 0.15, 0.3]); cb.ax.yaxis.set_major_formatter(PCT)
    cb.outline.set_linewidth(0.4); cb.ax.tick_params(labelsize=5.8, length=2)
    cb.set_label(r"input gain, % of $L(f_0)$", fontsize=6.3)
    fig.subplots_adjust(left=0.17, right=0.91, top=0.97, bottom=0.12)
    return fig


fig = fig_horizon_surprise()
V.save(fig, "rq4c_horizon_by_surprise")
plt.show()

---
## RQ5 — Quanto as conclusões dependem de como a incerteza é medida?

### 5A · Migração dos p-valores
Cada coluna é uma célula, para o ganho input. O eixo y é $-\log_{10} p$, então mais alto significa mais evidência.
- **Marcadores:**
  - círculo vazado = p só por datas;
  - pontos à direita = refit+datas, em cinza antes do Addendum E e coloridos com 2.000 sorteios;
  - quadrado = Holm.
- **Seta:** liga o p só por datas ao p de Holm.
- **Linhas horizontais:** a vermelha é α = 0,05; as pontilhadas são os pisos de resolução de cada número de sorteios.
- **Faixa rosada:** a região que o Holm não podia alcançar com 200 sorteios (12 células em 5/10 sessões, 6 em 30 dias). Ela começa abaixo de α, que é o defeito corrigido pelo Addendum E.
- **Leitura:** todas as células que eram significativas só por datas caem abaixo de α quando entram o refit e a multiplicidade. Com 2.000 sorteios nenhuma está presa a um piso.

In [ ]:
def fig_pmigration():
    items = V.cell_grid(V.SPECS, gap_asset=0.45, gap_spec=1.1)
    cC = V.claims_C().set_index(["asset", "family"])
    cD = V.claims_D().set_index(["asset", "spec", "family"])
    L = lambda p: -np.log10(np.clip(np.asarray(p, float), 1e-4, 1))
    top = 3.3
    fig, ax = plt.subplots(figsize=(V.W, 2.85))
    for s in V.SPECS:
        xs = [x for x, s_, _, _ in items if s_ == s]
        cells = 6 if s == "primary" else 12
        y0 = L(cells * 2 / 201)
        ax.add_patch(Rectangle((min(xs) - 0.5, y0), max(xs) - min(xs) + 1.0, top - y0, fc="#FBE6E2", ec="none", zorder=0))
    ax.axhline(L(0.05), color="#C0392B", lw=0.9, zorder=1)
    for B in (200, 500, 2000):
        ax.axhline(L(2 / (B + 1)), color="#8A8A8A", lw=0.5, ls=(0, (1, 1.8)), zorder=1)
        ax.text(items[-1][0] + 0.65, L(2 / (B + 1)), f"floor\nB = {B:,}", fontsize=5, color="#666666", va="center",
                ha="left", linespacing=0.9)
    ax.text(items[-1][0] + 0.65, L(0.05), "α = .05", fontsize=5.6, color="#C0392B", va="center", ha="left")
    for x, s, a, f in items:
        col = V.FAM_COLOR[f]
        p_date = V.boot_stat(a, s, f, "gain_input").p_zero
        p_pre = V.refit_summary(a, s, f, "gain_input", pre=True).p_zero
        p_post = V.refit_summary(a, s, f, "gain_input").p_zero
        p_holm = cC.loc[(a, f)].input_p_holm if s == "primary" else cD.loc[(a, s, f)].input_p_holm
        ax.add_patch(FancyArrowPatch((x, L(p_date)), (x, L(p_holm)), arrowstyle="-|>,head_length=2.2,head_width=1.1",
                                     color=col, lw=0.8, alpha=0.7, shrinkA=2.5, shrinkB=2.8, zorder=2))
        ax.plot(x, L(p_date), "o", ms=4, mfc="white", mec=col, mew=0.9, zorder=4)
        ax.plot(x + 0.24, L(p_pre), "o", ms=2.2, color="#A0A0A0", zorder=4)
        ax.plot(x + 0.24, L(p_post), "o", ms=2.9, color=col, zorder=5)
        ax.plot(x, L(p_holm), "s", ms=3.6, mfc=col, mec="white", mew=0.5, zorder=5)
    ticks = [1, 0.3, 0.1, 0.05, 0.01, 0.001]
    ax.set_yticks(L(ticks), ["1", ".3", ".1", ".05", ".01", ".001"])
    ax.set_ylim(-0.08, top)
    ax.set_xlim(items[0][0] - 0.6, items[-1][0] + 0.6)
    ax.set_ylabel("p-value of the input gain\n(log scale)", fontsize=7)
    V.horizon_separators(ax, items)
    V.multilevel_xlabels(ax, items, y_asset=-0.1, y_hor=-0.19)
    handles = [Line2D([], [], marker="o", ls="", mfc="white", mec="#555555", ms=4, label="date-only"),
               Line2D([], [], marker="o", ls="", color="#A0A0A0", ms=2.4, label="refit+date, before Add. E"),
               Line2D([], [], marker="o", ls="", color="#555555", ms=3, label="refit+date, 2,000 draws"),
               Line2D([], [], marker="s", ls="", mfc="#555555", mec="white", ms=3.6, label="Holm-adjusted"),
               Patch(fc="#FBE6E2", label="beyond the Holm floor with 200 draws")]
    ax.legend(handles=handles, loc="lower center", ncol=5, bbox_to_anchor=(0.5, 1.0), fontsize=6, columnspacing=1.0,
              handletextpad=0.3)
    fig.subplots_adjust(left=0.1, right=0.93, top=0.9, bottom=0.22)
    return fig


fig = fig_pmigration()
V.save(fig, "rq5a_pvalue_migration")
plt.show()

### 5B · Inflação dos intervalos e caudas catastróficas
- **(a) Inflação:** para cada célula, quanto o IC de 95% do ganho input se alarga quando a incerteza do refit entra, como razão entre as larguras, em escala log.
- **(b) Caudas do HAR:** a distribuição acumulada (ECDF) dos sorteios refit+datas do ganho input no SPX em 5 sessões, com eixo *symlog*. As barras embaixo são os ICs só por datas.
- **Leitura:**
  - o refit alarga os intervalos de ≈1× (BT) a mais de 20× (HAR);
  - no HAR com IV, uma fração não trivial dos reajustes é **catastrófica** (ganho input abaixo de −50%). Essa fragilidade é invisível ao bootstrap de datas.

In [ ]:
def fig_inflation():
    items = V.cell_grid(V.SPECS, gap_asset=0.45, gap_spec=1.1)
    fig = plt.figure(figsize=(V.W, 2.6))
    gs = GridSpec(1, 2, figure=fig, width_ratios=[2.0, 1], wspace=0.22, left=0.075, right=0.995, top=0.88, bottom=0.25)
    axL, axR = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])
    axL.axhline(1, color="#7A7A7A", lw=0.6)
    for x, s, a, f in items:
        b = V.boot_stat(a, s, f, "gain_input")
        rf = V.refit_summary(a, s, f, "gain_input")
        ratio = (rf.ci_hi - rf.ci_lo) / (b.ci_hi - b.ci_lo)
        axL.plot([x, x], [1, ratio], color=V.FAM_COLOR[f], lw=1.0, alpha=0.6, zorder=2)
        axL.plot(x, ratio, "o", ms=3.8, mfc=V.FAM_COLOR[f], mec="white", mew=0.5, zorder=3)
    axL.set_yscale("log")
    axL.yaxis.set_major_formatter(mt.FuncFormatter(lambda v, _: f"{v:g}×"))
    axL.set_ylim(0.8, 30)
    axL.set_xlim(items[0][0] - 0.6, items[-1][0] + 0.6)
    axL.set_ylabel("CI width: refit+date\n/ date-only", fontsize=7)
    axL.set_title("a   Interval inflation (input gain)", fontsize=7.6)
    V.horizon_separators(axL, items)
    V.multilevel_xlabels(axL, items, y_asset=-0.1, y_hor=-0.19, fs=5.4)
    V.soft_grid(axL, "y")
    a, s = "SPX", "h5td"
    axR.axvspan(-20, -0.5, color="#FBE3E1", lw=0, zorder=0)
    for k, f in enumerate(V.FAMILIES):
        v = np.sort(V.refit_draws(a, s, f)["rel_gain_input"].to_numpy(float))
        axR.step(v, np.arange(1, v.size + 1) / v.size, where="post", color=V.FAM_COLOR[f], lw=1.2, zorder=3,
                 label=f"{V.FAM_SHORT[f]}: {(v < -0.5).mean():.1%} catastrophic")
        b = V.boot_stat(a, s, f, "rel_gain_input")
        axR.plot([b.ci_lo, b.ci_hi], [-0.06 - 0.04 * k] * 2, color=V.FAM_COLOR[f], lw=2, solid_capstyle="butt")
    axR.text(-1.4, 0.92, "input gain\n< −50%", ha="center", va="top", fontsize=5.6, color="#A04030")
    axR.text(0.55, -0.1, "date-only\n95% CIs", fontsize=5.2, color="#555555", ha="left", va="center")
    axR.set_xscale("symlog", linthresh=0.05, linscale=1.2)
    axR.set_xlim(-12, 1.0); axR.set_ylim(-0.2, 1.03)
    axR.set_xticks([-10, -1, -0.1, 0, 0.1, 1], ["−1000%", "−100%", "−10%", "0", "10%", "100%"], fontsize=5.8)
    axR.axvline(0, color="#7A7A7A", lw=0.5)
    axR.set_ylabel("ECDF", fontsize=7)
    axR.set_xlabel(r"input gain, % of $L(f_0)$  (SPX, 5 sessions)", fontsize=6.6)
    axR.set_title("b   Refit tails", fontsize=7.6)
    axR.legend(loc="center left", bbox_to_anchor=(0.0, 0.55), fontsize=5.6, handlelength=1.2)
    V.soft_grid(axR, "y")
    return fig


fig = fig_inflation()
V.save(fig, "rq5b_interval_inflation")
plt.show()

### 5C · Painel de calibração da simulação
Os resultados vêm da auditoria Monte Carlo, com 500 réplicas por desenho e verdade conhecida.
- **(a) Cobertura:** para cada componente (teacher, input, total) e cada learner, o ponto é a cobertura média dos ICs de 95% (bootstrap de datas, lei A) e a barra vai do pior ao melhor dos cinco desenhos. A faixa cinza é a banda de Monte Carlo em torno de 95%.
- **(b) e (c) Taxas de disparo:** com que frequência as regras C1 (ganho input) e C2 (teacher além do placebo) disparam em cada desenho, do redundante ao informativo. A linha tracejada é o limite de 10% de falsa detecção.
- **Leitura:**
  - a C1 tem bom tamanho (0–3% de falsa detecção no desenho redundante) e potência alta para sinais fortes (83–100%), mas baixa para sinais fracos;
  - os ICs do **teacher** cobrem bem abaixo do nominal, o que justifica não confiar só no bootstrap de datas.

In [ ]:
def fig_simulation():
    S = pd.read_csv(V.PRI / "simulation" / "summary_teacher.csv")
    S = S[S.law == "A_wellspec"]
    learners = ["oracle", "har", "enet_light", "enet_heavy", "xgb"]
    LL = {"oracle": "Oracle", "har": "HAR", "enet_light": "EN light", "enet_heavy": "EN heavy", "xgb": "BT"}
    LC = {"oracle": "#555555", "har": V.FAM_COLOR["har"], "enet_light": "#F2A65A", "enet_heavy": V.FAM_COLOR["enet"],
          "xgb": V.FAM_COLOR["xgb"]}
    designs = [("redundant", 0.0), ("constant", 0.1), ("constant", 0.25), ("state", 0.1), ("state", 0.25)]
    dlab = ["redun-\ndant", "const\n.10", "const\n.25", "state\n.10", "state\n.25"]
    fig = plt.figure(figsize=(V.W, 2.55))
    gs = GridSpec(1, 3, figure=fig, width_ratios=[1.1, 1, 1], wspace=0.3, left=0.075, right=0.86, top=0.88, bottom=0.22)
    ax = fig.add_subplot(gs[0])
    band = 1.96 * np.sqrt(0.95 * 0.05 / 500)
    ax.axhspan(0.95 - band, 0.95 + band, color="#E6E6E6", lw=0, zorder=0)
    ax.axhline(0.95, color="#7A7A7A", lw=0.6, zorder=1)
    for c, comp in enumerate(("teacher", "input", "total")):
        for k, lr in enumerate(learners):
            v = S[S.learner == lr][f"{comp}_covers_pop"].to_numpy(float)
            x = c + (k - 2) * 0.13
            ax.plot([x, x], [v.min(), v.max()], color=LC[lr], lw=1.0, zorder=2)
            ax.plot(x, v.mean(), "o", ms=3.4, mfc=LC[lr], mec="white", mew=0.5, zorder=3)
    ax.set_xticks([0, 1, 2], ["teacher", "input", "total"], fontsize=6.5)
    for lab, comp in zip(ax.get_xticklabels(), ("teacher", "input", "total")):
        lab.set_color(V.COMP_COLOR[comp])
    ax.tick_params(axis="x", length=0)
    ax.set_ylim(0.5, 1.0)
    ax.yaxis.set_major_formatter(PCT)
    ax.set_ylabel("coverage of the\npopulation gain", fontsize=7)
    ax.set_title("a   95% CI coverage", fontsize=7.4)
    V.soft_grid(ax, "y")
    for j, (rule, name) in enumerate((("detect_C1", "b   C1: input claimed"), ("detect_C2", "c   C2: teaches"))):
        ax = fig.add_subplot(gs[j + 1])
        ax.axhline(0.10, color="#C0392B", lw=0.7, ls=(0, (3, 2)), zorder=1)
        ax.axvspan(-0.4, 0.4, color="#F4F4F4", lw=0, zorder=0)
        for lr in learners:
            r = S[S.learner == lr]
            vals = [r[(r["mode"] == m) & np.isclose(r.b_bar, b)][rule].iloc[0] for m, b in designs]
            if np.all(np.isnan(vals)):
                continue
            ax.plot(range(len(designs)), vals, "-o", color=LC[lr], lw=1.1, ms=3, mfc="white", mew=0.9, zorder=3)
        ax.set_xticks(range(len(designs)), dlab, fontsize=5.6)
        ax.tick_params(axis="x", length=0)
        ax.set_ylim(-0.03, 1.03)
        ax.yaxis.set_major_formatter(PCT)
        ax.set_title(name, fontsize=7.4)
        ax.text(4.3, 0.115, "10%", fontsize=5.4, color="#C0392B", ha="right", va="bottom")
        V.soft_grid(ax, "y")
        if j == 0:
            ax.set_ylabel("share of replicates", fontsize=7)
    handles = [Line2D([], [], color=LC[l], marker="o", ms=3, lw=1.0, label=LL[l]) for l in learners]
    fig.legend(handles=handles, loc="center left", bbox_to_anchor=(0.865, 0.55), fontsize=6.2, title="learner",
               title_fontsize=6.2)
    fig.text(0.5, 0.02, "design: redundant = V adds nothing; const / state = constant or state-dependent signal "
             "strength b", ha="center", fontsize=5.6, color="#555555")
    return fig


fig = fig_simulation()
V.save(fig, "rq5c_simulation_board")
plt.show()

---
### Arquivos gerados

In [ ]:
for p in sorted(V.FIG.glob("*.pdf")):
    print(f"{p.name:42s}  pdf {p.stat().st_size / 1e3:7.0f} kB   png {p.with_suffix('.png').stat().st_size / 1e6:5.1f} MB")